In [8]:
import polars as pl
import os

from pathlib import Path

In [9]:
WORKING_PATH = Path('/group/pmc021/amunif/epi-thesis/workflow/17_Pairwise Ranking HepG2 GSE76344/')
DATASET_PATH = WORKING_PATH / 'dataset'
OUTPUT_PATH  = WORKING_PATH / 'output' / 'combined'

In [13]:
### Merge the results into single CSV file

# Read all CSV into single dataframe
pl_df = pl.read_csv(OUTPUT_PATH / 'test' / "*-test-metrics.csv")

In [14]:
pl_df

model,seed,histone_marker,epochs_trained,val_accuracy,val_auc,test_accuracy,test_auc,test_precision,test_recall,test_f1,antisymmetry
str,i64,str,i64,f64,f64,f64,f64,f64,f64,f64,f64
"""DirectRanker""",1011,"""H3K4me3""",42,67.5,0.77,67.2,0.7593,0.6434,0.7231,0.6809,0.903
"""LogisticRegression""",1011,"""H3K4me3""",null,69.5,0.7562,70.5,0.7447,0.7234,0.6322,0.6748,0.752
"""RandomForest""",1011,"""H3K4me3""",null,70.3,0.7907,69.5,0.775,0.7039,0.6384,0.6696,0.741
"""SVM_Linear""",1011,"""H3K4me3""",null,69.3,0.756,70.6,0.745,0.7251,0.6322,0.6755,0.751
"""DirectRanker""",123,"""H3K4me3""",46,66.3,0.74,68.3,0.7495,0.6508,0.7386,0.6919,0.902
…,…,…,…,…,…,…,…,…,…,…,…
"""SVM_Linear""",456,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,66.7,0.7301,64.7,0.6934,0.6401,0.5903,0.6142,0.702
"""DirectRanker""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",100,68.1,0.75,70.8,0.7875,0.681,0.7296,0.7045,0.976
"""LogisticRegression""",789,"""H3K4me3-H3K9ac-H3K9me3-H3K27ac…",null,65.5,0.7029,68.3,0.7319,0.6802,0.6331,0.6558,0.701


In [15]:
summary_df = (
    pl_df
    .group_by(["histone_marker", "model"])
    .agg([
        pl.col("val_accuracy").mean().alias("val_accuracy_mean"),
        pl.col("val_accuracy").std().alias("val_accuracy_std"),
        pl.col("val_auc").mean().alias("val_auc_mean"),
        pl.col("val_auc").std().alias("val_auc_std"),
        pl.col("test_accuracy").mean().alias("test_accuracy_mean"),
        pl.col("test_accuracy").std().alias("test_accuracy_std"),
        pl.col("test_auc").mean().alias("test_auc_mean"),
        pl.col("test_auc").std().alias("test_auc_std"),
    ])
    .sort(["histone_marker", "test_accuracy_mean"], descending=[False, True])
    .with_columns(pl.col(pl.Float64).round(4))
)

In [18]:
summary_df

histone_marker,model,val_accuracy_mean,val_accuracy_std,val_auc_mean,val_auc_std,test_accuracy_mean,test_accuracy_std,test_auc_mean,test_auc_std
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""H3K27ac""","""RandomForest""",70.16,1.6134,0.7778,0.0214,70.9,1.9545,0.7828,0.0197
"""H3K27ac""","""LogisticRegression""",68.4,1.8398,0.7303,0.024,68.22,1.63,0.7266,0.0215
"""H3K27ac""","""SVM_Linear""",68.38,1.8472,0.7309,0.0239,68.1,2.1783,0.7276,0.0208
"""H3K27ac""","""DirectRanker""",67.76,2.2546,0.758,0.0239,67.6,1.2629,0.7612,0.0215
"""H3K27ac-H3K27me3""","""RandomForest""",70.32,2.4641,0.7803,0.0219,70.46,2.3115,0.7861,0.0224
…,…,…,…,…,…,…,…,…,…
"""H3K9me3-H3K27ac-H3K27me3""","""SVM_Linear""",66.12,1.8647,0.711,0.023,66.3,2.1691,0.7075,0.0206
"""H3K9me3-H3K27me3""","""RandomForest""",65.04,2.305,0.6996,0.0198,65.92,2.2554,0.7088,0.0308
"""H3K9me3-H3K27me3""","""DirectRanker""",64.04,1.4639,0.7,0.02,64.26,2.3586,0.7088,0.0334


In [19]:
summary_df.write_csv(OUTPUT_PATH/ f"HepG2 GSE76344 Ranking.csv", include_header=True)